In [ ]:
from util.text_util import join_texts
%load_ext autoreload
%autoreload 2

# Load the BPMN Dataset

In [ ]:
from mcp4cm.bpmn.dataloading.dataloading import BPMNModelCollection
from mcp4cm.bpmn.dataloading.bpmn_dataset import BPMNDataset
from mcp4cm.dataloading import load_dataset
from mcp4cm.base import DatasetType

Defining file path for saving dataset and loading it later on.

In [ ]:
bpmn_dataset = load_dataset(path='data/bpmnmodelset', dataset_type=DatasetType.BPMNMODELSET, bpmn_model_collection=BPMNModelCollection.BPMAI)

In [ ]:
from mcp4cm.bpmn.data_extraction import extract_names_from_models
from mcp4cm.bpmn.constants import NAMES_WITH_TYPES_COLUMN, NAMES_COLUMN, EMPTY_NAME_TOKEN

use_types = True
empty_name = EMPTY_NAME_TOKEN

if use_types:
    key = NAMES_WITH_TYPES_COLUMN
    file_path = 'data/bpmnmodelset/processed/culled_with_typed_names.csv'
else:
    key = NAMES_COLUMN
    file_path = 'data/bpmnmodelset/processed/culled_with_names.csv'


extract_names_from_models(bpmn_dataset, use_types=use_types, include_texts=True, include_documentation=True)


In [ ]:

from bpmn.filter_functions import filter_empty_models, filter_models_by_min_element_count, \
    filter_models_by_max_element_count, filter_models_by_element_count, filter_models_by_empty_name_percentage, \
    filter_models_by_dummy_words, filter_models_by_median_name_length, filter_models_by_required_elements, \
    filter_models_by_duplicate_activities
from mcp4cm.dataloading import Dataset

print(bpmn_dataset)
Dataset.apply_filters(
    dataset=bpmn_dataset,
    filters=[
        filter_empty_models,
        filter_models_by_empty_name_percentage,
        filter_models_by_min_element_count,
        filter_models_by_required_elements,
        filter_models_by_dummy_words,
        filter_models_by_median_name_length,
        filter_models_by_duplicate_activities
    ],
)

print(bpmn_dataset)

In [ ]:
from mcp4cm.language_detection import extract_dataset_languages, get_models_by_language

extract_dataset_languages(bpmn_dataset, text_key=key, empty_name=empty_name, override=False)


In [ ]:
#language_dict = get_models_by_language(bpmn_dataset, print_counts=True)

In [ ]:
from mcp4cm.language_detection import filter_models_by_language
english_dataset = filter_models_by_language(bpmn_dataset, 'en', key=key, empty_name=empty_name)

#language_dict = get_models_by_language(english_dataset)
print(f"Filtered Dataset to {len(english_dataset)} English models.")

del bpmn_dataset



In [ ]:
file_path = 'data/bpmnmodelset/processed/bpmai/english_typed_models.csv'
BPMNDataset.to_csv(english_dataset, file_path);
print(f"Saved {len(english_dataset)} models in dataset {english_dataset.name} to csv file")


In [ ]:
from mcp4cm.duplicate_detection import detect_duplicates_by_hash
hash_unique, hash_duplicate = detect_duplicates_by_hash(english_dataset, key=key, inplace=False, plt_fig=True, print_results=True, keep_one=True)


In [ ]:
from mcp4cm.duplicate_detection import tfidf_near_duplicate_detector
unique_dataset, duplicate_dataset = tfidf_near_duplicate_detector(english_dataset,key=key, threshold=0.95, inplace=False, plt_fig=True, print_results=True, keep_one=True)

In [ ]:
file_path = 'data/bpmnmodelset/processed/bpmai/english_typed_deduplicated_models.csv'
BPMNDataset.to_csv(unique_dataset, file_path)

BPMNDataset.to_files(unique_dataset, output_directory='data/export/bpmai_typed_clean', include_svg=True)

In [ ]:
#duplicate_file_path = 'data/bpmnmodelset/processed/bpmai/duplicate_models.csv'
#BPMNDataset.to_csv(duplicate_dataset, duplicate_file_path);